# 03 — Build Baseline Dataset

This notebook constructs the final analysis-ready dataset used throughout
the study.

The notebook performs three tasks:

1. Compute baseline objective values for each instance.
2. Merge baseline information with structural graph properties.
3. Export the final dataset used by all downstream analyses.

Inputs:

- final_dataset.parquet
- baseline.csv
- graph_metrics.csv

Outputs:

- baseline_enriched.csv
- baseline_dataset.parquet

In [10]:
# Imports
import sys
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx

from tqdm import tqdm


# Project Imports
sys.path.append(str(Path("../src/maxcut").resolve()))

%load_ext autoreload
%autoreload 2

import config as cfg
import utils

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# =========================================================
# Input Files
# =========================================================

PATH_RESULTS = (
    cfg.PROCESSED_DIR / "final_dataset.parquet"
)

PATH_BASELINE = (
    cfg.DATA_DIR / "metadata" / "baseline.csv"
)

PATH_GRAPH_METRICS = (
    cfg.PROCESSED_DIR / "graph_metrics.csv"
)

# =========================================================
# Output Files
# =========================================================

PATH_BASELINE_ENRICHED = (
    cfg.PROCESSED_DIR / "baseline_enriched.csv"
)

PATH_BASELINE_DATASET = (
    cfg.PROCESSED_DIR / "baseline_dataset.csv"
)

In [3]:
df_results = pd.read_parquet(PATH_RESULTS)

df_baseline = pd.read_csv(PATH_BASELINE)

df_metrics = pd.read_csv(PATH_GRAPH_METRICS)

print(df_results.shape)
print(df_baseline.shape)
print(df_metrics.shape)

(115945, 7)
(3506, 7)
(3506, 16)


In [4]:
df_results.head()

,file,name,heuristic,seed,objective,runtime,solution
0,./BURER2002/G55-13.945-5.csv,G55,BURER2002,144,10243.0,13.964832,1110100000010001111111001011010010000001110001...
1,./BURER2002/G55-13.945-5.csv,G55,BURER2002,244,10247.0,13.946340,0000110011100100000100110000001001001111001100...
2,./BURER2002/G55-13.945-5.csv,G55,BURER2002,344,10249.0,13.960023,1010010111111000110000000101000001011100010100...
3,./BURER2002/G55-13.945-5.csv,G55,BURER2002,444,10243.0,13.963533,0001011001100011010000101111000101101000000110...
4,./BURER2002/G55-13.945-5.csv,G55,BURER2002,544,10233.0,13.970771,0011110101110000011100011000010001001111011011...


In [5]:
df_results['name'].nunique()

3515

In [9]:
# removing instances with trivial node count whose values for other solvers are not available
small_graphs = df_baseline[df_baseline["nodes"] <= 20]

print(f"Instances with n <= 20: {len(small_graphs)}")

display(
    small_graphs["nodes"]
    .value_counts()
    .sort_index()
)

print(
    f"Total instances: {len(df_baseline)}"
)

print(
    f"Fraction removed: {100*len(small_graphs)/len(df_baseline):.2f}%"
)

df_baseline = df_baseline[df_baseline["nodes"] > 20]

print(
    f"New Total instances: {len(df_baseline)}"
)

Instances with n <= 20: 106


nodes
3      4
4      5
5      6
6      7
7      3
8     12
9      3
10     7
11     7
12     7
13     5
14     6
15     1
16     5
17     8
18     6
20    14
Name: count, dtype: int64

Total instances: 3506
Fraction removed: 3.02%
New Total instances: 3400


In [11]:
df_baseline["size_cat"] = df_baseline["nodes"].apply(utils.categorize_size)
df_baseline["density_cat"] = df_baseline["density"].apply(utils.categorize_density)

In [12]:
df_baseline.head()

,name,nodes,edges,density,limit,has_leafs,int_only,size_cat,density_cat
0,G55,4969,12498,0.001013,13.945,True,True,large,sparse
1,G56,4969,12498,0.001013,13.922,True,True,large,sparse
2,G57,5000,10000,0.000800,11.299,False,True,large,sparse
3,G58,5000,29570,0.002366,11.876,False,True,large,sparse
4,G59,5000,29570,0.002366,15.875,False,True,large,sparse


In [13]:
conditions = [
    (df_baseline["size_cat"].isin(["xsmall", "small", "medium", "large"])) &
    (df_baseline["density_cat"] == "sparse"),

    (df_baseline["size_cat"] == "xlarge") &
    (df_baseline["density_cat"] == "sparse")
]

choices = [
    "BURER2002",
    "MERZ1999GLS"
]

df_baseline["heuristics_name"] = np.select(
    conditions,
    choices,
    default="PALUBECKIS2004bMST2"
)

In [14]:
dav2_df = df_results[df_results["heuristic"] == "DAv2"].copy()
idx = dav2_df.groupby("name")["objective"].idxmax()
best_dav2 = dav2_df.loc[idx, ["name", "objective",'runtime']]
best_dav2 = best_dav2.rename(columns={
    "objective": "DAv2",
    'runtime': 'DAv2_runtime'
})
df_baseline = df_baseline.merge(best_dav2, on="name", how="left")

In [15]:
dav3_df = df_results[df_results["heuristic"] == "DAv3"].copy()
idx = dav3_df.groupby("name")["objective"].idxmax()
best_dav3 = dav3_df.loc[idx, ["name", "objective",'runtime']]
best_dav3 = best_dav3.rename(columns={
    "objective": "DAv3",
    'runtime': 'DAv3_runtime'
})
df_baseline = df_baseline.merge(best_dav3, on="name", how="left")

In [16]:
dav3c_df = df_results[df_results["heuristic"] == "DAv3c"].copy()
idx = dav3c_df.groupby("name")["objective"].idxmax()
best_dav3c = dav3c_df.loc[idx, ["name", "objective",'runtime']]
best_dav3c = best_dav3c.rename(columns={
    "objective": "DAv3c",
    'runtime': 'DAv3c_runtime'
})
df_baseline = df_baseline.merge(best_dav3c, on="name", how="left")

In [17]:
idx = (
    df_results[df_results["heuristic"].notna()]
    .groupby(["name", "heuristic"])["objective"]
    .idxmax()
)

best_heuristic = (
    df_results.loc[
        idx,
        ["name", "heuristic", "objective", "runtime"]
    ]
    .rename(
        columns={
            "objective": "heuristic objective",
            "runtime": "heuristic_runtime"
        }
    )
)

In [18]:
df_baseline = df_baseline.merge(
    best_heuristic,
    left_on=["name", "heuristics_name"],
    right_on=["name", "heuristic"],
    how="left"
)
df_baseline = df_baseline.drop("heuristic", axis=1)
df_baseline = df_baseline.rename(columns={"heuristic objective": "heuristic"})

In [19]:
df_baseline.head()

,name,nodes,edges,density,limit,has_leafs,int_only,size_cat,density_cat,heuristics_name,DAv2,DAv2_runtime,DAv3,DAv3_runtime,DAv3c,DAv3c_runtime,heuristic,heuristic_runtime
0,G55,4969,12498,0.001013,13.945,True,True,large,sparse,BURER2002,10294.0,13.773063,10296.0,16.403668,10200.0,13.121273,10249.0,13.960023
1,G56,4969,12498,0.001013,13.922,True,True,large,sparse,BURER2002,4008.0,13.813462,4015.0,12.644328,3955.0,13.119173,3982.0,13.931711
2,G57,5000,10000,0.000800,11.299,False,True,large,sparse,BURER2002,3468.0,11.263977,3484.0,13.233151,3452.0,11.140920,3424.0,11.312191
3,G58,5000,29570,0.002366,11.876,False,True,large,sparse,BURER2002,19258.0,11.820511,19269.0,12.885950,19178.0,11.147025,19092.0,11.924794
4,G59,5000,29570,0.002366,15.875,False,True,large,sparse,BURER2002,6062.0,15.794570,6077.0,16.119326,5981.0,15.144516,5915.0,15.877029


In [20]:
# =========================================================
# Validation
# =========================================================

print(
    "Instances:",
    len(df_baseline)
)

print(
    "Missing heuristic objectives:",
    df_baseline["heuristic"]
    .isna()
    .sum()
)

print(
    "Missing heuristic runtimes:",
    df_baseline["heuristic_runtime"]
    .isna()
    .sum()
)

print(
    "Missing DAv3 objectives:",
    df_baseline["DAv3"]
    .isna()
    .sum()
)

print(
    "Missing DAv3 runtimes:",
    df_baseline["DAv3_runtime"]
    .isna()
    .sum()
)

print(
    "Missing Dav3c objectives:",
    df_baseline["DAv3c"]
    .isna()
    .sum()
)

print(
    "Missing DAv3c runtimes:",
    df_baseline["DAv3c_runtime"]
    .isna()
    .sum()
)

print(
    "Missing DAv2 objectives:",
    df_baseline["DAv2"]
    .isna()
    .sum()
)

print(
    "Missing DAv2 runtimes:",
    df_baseline["DAv2_runtime"]
    .isna()
    .sum()
)

print(
    "Missing DAv2 instances size category:",
    df_baseline[df_baseline["DAv2_runtime"]
    .isna()]['size_cat'].value_counts()  
)

Instances: 3400
Missing heuristic objectives: 0
Missing heuristic runtimes: 0
Missing DAv3 objectives: 0
Missing DAv3 runtimes: 0
Missing Dav3c objectives: 0
Missing DAv3c runtimes: 0
Missing DAv2 objectives: 81
Missing DAv2 runtimes: 81
Missing DAv2 instances size category: size_cat
xlarge    81
Name: count, dtype: int64


In [21]:
df_baseline.to_csv(
    PATH_BASELINE_ENRICHED,
    index=False
)

print(
    f"Saved to:\n{PATH_BASELINE_ENRICHED}"
)

Saved to:
/Users/sargun/Desktop/DAMB-DAv3c/data/processed/baseline_enriched.csv


In [22]:
df_baseline["baseline_objective"] = (
    df_baseline[cfg.SOLVERS]
    .max(axis=1)
)

In [23]:
def get_best_solvers(row):

    best = row["baseline_objective"]

    solvers = []

    # DAv2
    if pd.notna(row["DAv2"]) and row["DAv2"] == best:
        solvers.append("DAv2")

    # DAv3
    if pd.notna(row["DAv3"]) and row["DAv3"] == best:
        solvers.append("DAv3")

    # DAv3c
    if pd.notna(row["DAv3c"]) and row["DAv3c"] == best:
        solvers.append("DAv3c")

    # Benchmark heuristic
    if (
        pd.notna(row["heuristic"])
        and row["heuristic"] == best
    ):
        solvers.append(
            row["heuristics_name"]
        )

    return ";".join(sorted(set(solvers)))

In [24]:
df_baseline["best_solvers"] = (
    df_baseline.apply(get_best_solvers, axis=1)
)

In [25]:
df_graph_metrics = pd.read_csv(PATH_GRAPH_METRICS)

In [26]:
df_graph_metrics.head()

,name,nodes,edges,bipartite,neg_edges,neg_edges_frac,neg_wt_sum,neg_wt_ratio,triangle_count,has_triangles,transitivity,average_clustering,cyclomatic_number,bridge_count,bridge_fraction,unique_weights
0,G55,5000,12498,False,0,0.000000,0.0,0.000000,23,True,0.001100,0.001189,7530,180,0.014402,1
1,G56,5000,12498,False,6276,0.502160,-6276.0,0.502160,23,True,0.001100,0.001189,7530,180,0.014402,2
2,G57,5000,10000,True,5019,0.501900,-5019.0,0.501900,0,False,0.000000,0.000000,5001,0,0.000000,2
3,G58,5000,29570,False,0,0.000000,0.0,0.000000,30784,True,0.098091,0.326822,24571,0,0.000000,1
4,G59,5000,29570,False,14737,0.498377,-14737.0,0.498377,30784,True,0.098091,0.326822,24571,0,0.000000,2


In [27]:
len(df_graph_metrics)

3506

In [28]:
# Columns appearing in both dataframes
common_cols = (
    set(df_baseline.columns)
    & set(df_graph_metrics.columns)
)

# Keep the join key
common_cols.discard("name")

print("Overlapping columns:")
print(sorted(common_cols))

Overlapping columns:
['edges', 'nodes']


In [29]:
df_graph_metrics_clean = (
    df_graph_metrics
    .drop(columns=list(common_cols))
)

In [30]:
df_baseline_dataset = (
    df_baseline
    .merge(
        df_graph_metrics_clean,
        on="name",
        how="left"
    )
)

In [31]:
print(df_baseline.shape)
print(df_graph_metrics.shape)
print(df_baseline_dataset.shape)

print(
    "Duplicate names:",
    df_baseline_dataset["name"]
    .duplicated()
    .sum()
)

print(
    "NaN values per column:",
    df_baseline_dataset.isna().sum().sort_values(ascending=False)
)

(3400, 20)
(3506, 16)
(3400, 33)
Duplicate names: 0
NaN values per column: DAv2_runtime          81
DAv2                  81
name                   0
neg_wt_ratio           0
bipartite              0
neg_edges              0
neg_edges_frac         0
neg_wt_sum             0
triangle_count         0
baseline_objective     0
has_triangles          0
transitivity           0
average_clustering     0
cyclomatic_number      0
bridge_count           0
bridge_fraction        0
best_solvers           0
heuristic              0
heuristic_runtime      0
nodes                  0
DAv3c_runtime          0
DAv3c                  0
DAv3_runtime           0
DAv3                   0
heuristics_name        0
density_cat            0
size_cat               0
int_only               0
has_leafs              0
limit                  0
density                0
edges                  0
unique_weights         0
dtype: int64


In [32]:
df_baseline_dataset.to_csv(
    PATH_BASELINE_DATASET,
    index=False
)

print(
    f"Saved to:\n{PATH_BASELINE_DATASET}"
)

Saved to:
/Users/sargun/Desktop/DAMB-DAv3c/data/processed/baseline_dataset.csv
